In [1]:
import numpy as np
from jaxtyping import Float
import pandas as pd
import matplotlib.pyplot as plt

# muutils
from muutils.jsonlines import jsonl_write, jsonl_load

# attention-motifs
from attention_motifs.bins import Bins
from attention_motifs.features.features import scalar_feature_table
from attention_motifs.features.hist_beta_fit import hist_beta_fit
from attention_motifs.util import prefix_dict
from attention_motifs.features.transition_tensor import tt_features
from attention_motifs.features.vec_features import vec_features
from attention_motifs.math.cos_sim import cosine_similarity_matrix
from attention_motifs.math.math import skew_lt

In [2]:
def gram_features(A: Float[np.ndarray, "n_ctx n_ctx"]) -> dict[str, float]:
	# dbg_tensor(A)
	return prefix_dict(
		hist_beta_fit(
			A.flatten(),
			bins=Bins(n_bins=32, start=0.0, stop=1.0),
		),
		prefix="beta_hist",
	)
	# TODO: mass as a function of distance from diagonal

In [3]:
def compute_scalar_features(
	A: Float[np.ndarray, "n_ctx n_ctx"],
) -> dict[str, float]:
	# dbg_tensor(A)
	A_log: Float[np.ndarray, "n_ctx n_ctx"] = np.nan_to_num(np.log(A + 1e-9), nan=-10)
	# dbg_tensor(A_log)

	A_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A)
	# dbg_tensor(A_skew)
	A_log_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A_log)

	return dict(
		# diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A.diagonal()), prefix="diag"),
		# off-diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A[:, 0]), prefix="first_tok"),
		# transition tensor: standard features, standard features on diff, linear envelope on transition time
		# 	TODO: standard features on decay rate
		**prefix_dict(
			tt_features(A),
			prefix="markov_transition",
		),
		# # {log, raw} gram matrix of {rows, cols, rows of skewed}: beta fit hist
		# # 	TODO: fit fft in `gram_features`, but this is expensive
		**prefix_dict(
			gram_features(A @ A.T),
			prefix=["gram", "row"],
		),
		**prefix_dict(
			gram_features(A.T @ A),
			prefix=["gram", "col"],
		),
		**prefix_dict(
			gram_features(A_skew.T @ A_skew),
			prefix=["gram", "skew"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log)),
			prefix=["log", "gram", "row"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log, col=True)),
			prefix=["log", "gram", "col"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log_skew)),
			prefix=["log", "gram", "skew"],
		),
	)

In [ ]:
df: pd.DataFrame = scalar_feature_table(
	features_func=compute_scalar_features,
	# models="pythia-14m".split(","),
)

models: ['pythia-14m', 'gemma-2b', 'gpt2-small', 'pythia-1b', 'gpt2-medium', 'tiny-stories-1M']
model: 'pythia-14m'
✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
128 prompts loaded


100%|██████████| 128/128 [04:53<00:00,  2.29s/it]

model: 'gemma-2b'
✔️  (0.00s) setting up paths                                                   
| (0.00s) loading prompts                                                      

✔️  (0.01s) loading prompts                                                    
128 prompts loaded


 12%|█▎        | 16/128 [04:49<21:02, 11.27s/it]  

In [ ]:
path: str = "../data/scalar_features.jsonl.gz"


In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:

jsonl_write(path, df.to_dict(orient="records"), use_gzip=True)

In [ ]:
_temp_loaded = pd.DataFrame(jsonl_load(path))
df = _temp_loaded

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
def plot_histograms_long(df: pd.DataFrame) -> None:
	"""Plot histograms for each feature with different models superimposed.

	This function assumes the DataFrame is in long format with columns:
	"model", "feat_name", "feat_val", and optionally "prompt", "layer", "head".

	# Parameters:
	 - `df : pd.DataFrame`
		 DataFrame containing the data.

	# Returns:
	 - `None`
		 Displays the histograms.
	"""
	features: list[str] = df["feat_name"].unique().tolist()
	models: list[str] = df["model"].unique().tolist()

	for feature in features:
		plt.figure()
		subset_feature: pd.DataFrame = df[df["feat_name"] == feature]
		for model in models:
			subset_model: pd.DataFrame = subset_feature[
				subset_feature["model"] == model
			]
			plt.hist(
				subset_model["feat_val"], bins=50, alpha=0.5, label=model, density=True
			)
		plt.xlabel(feature)
		plt.ylabel("Frequency")
		plt.title(f"Histogram of {feature} for different models")
		plt.legend()
		plt.show()


plot_histograms_long(df)